In [1]:
!pip install -q keras-tuner

In [2]:
import tensorflow as tf
import keras_tuner as kt

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import regularizers
from tensorflow.keras import callbacks

import numpy as np
import matplotlib.pyplot as plt

2026-06-15 16:20:23.463503: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781540423.712517      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781540423.785636      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781540424.399058      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781540424.399101      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781540424.399104      58 computation_placer.cc:177] computation placer alr

In [3]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

# Normalize
train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

# Reshape for CNN
train_images = train_images.reshape((-1,28,28,1))
test_images = test_images.reshape((-1,28,28,1))

# One Hot Encoding
train_labels = tf.keras.utils.to_categorical(train_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.02),
    layers.RandomZoom(0.05),
    layers.RandomTranslation(0.05,0.05)
])

I0000 00:00:1781540449.971168      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [5]:
def build_model(hp):

    model = models.Sequential()

    model.add(layers.Input(shape=(28,28,1)))

    model.add(data_augmentation)

    # ---------------------
    # CNN Block 1
    # ---------------------

    model.add(
        layers.Conv2D(
            filters=hp.Choice(
                "filters_1",
                values=[32,64]
            ),
            kernel_size=(3,3),
            activation='relu',
            padding='same'
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D())

    model.add(
        layers.Dropout(
            hp.Float(
                "dropout_1",
                min_value=0.1,
                max_value=0.4,
                step=0.1
            )
        )
    )

    # ---------------------
    # CNN Block 2
    # ---------------------

    model.add(
        layers.Conv2D(
            filters=hp.Choice(
                "filters_2",
                values=[64,128]
            ),
            kernel_size=(3,3),
            activation='relu',
            padding='same'
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D())

    model.add(
        layers.Dropout(
            hp.Float(
                "dropout_2",
                min_value=0.1,
                max_value=0.4,
                step=0.1
            )
        )
    )

    # ---------------------
    # CNN Block 3
    # ---------------------

    model.add(
        layers.Conv2D(
            filters=hp.Choice(
                "filters_3",
                values=[64,128,256]
            ),
            kernel_size=(3,3),
            activation='relu',
            padding='same'
        )
    )

    model.add(layers.BatchNormalization())

    model.add(layers.Flatten())

    # ---------------------
    # Dense Layer
    # ---------------------

    model.add(
        layers.Dense(
            units=hp.Int(
                "dense_units",
                min_value=64,
                max_value=256,
                step=64
            ),
            activation='relu',
            kernel_regularizer=regularizers.l2(
                hp.Choice(
                    "l2",
                    values=[1e-4,1e-3,1e-2]
                )
            )
        )
    )

    model.add(
        layers.Dropout(
            hp.Float(
                "dense_dropout",
                min_value=0.2,
                max_value=0.5,
                step=0.1
            )
        )
    )

    model.add(
        layers.Dense(
            10,
            activation='softmax'
        )
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=hp.Choice(
            "learning_rate",
            values=[1e-2,1e-3,5e-4,1e-4]
        )
    )

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [6]:
tuner = kt.BayesianOptimization(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=10,
    num_initial_points=5,
    seed=42,
    overwrite=True,
    directory='cnn_tuner',
    project_name='mnist_cnn'
)

In [7]:
tuner.search_space_summary()

Search space summary
Default search space size: 9
filters_1 (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64], 'ordered': True}
dropout_1 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.4, 'step': 0.1, 'sampling': 'linear'}
filters_2 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128], 'ordered': True}
dropout_2 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.4, 'step': 0.1, 'sampling': 'linear'}
filters_3 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256], 'ordered': True}
dense_units (Int)
{'default': None, 'conditions': [], 'min_value': 64, 'max_value': 256, 'step': 64, 'sampling': 'linear'}
l2 (Choice)
{'default': 0.0001, 'conditions': [], 'values': [0.0001, 0.001, 0.01], 'ordered': True}
dense_dropout (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01

In [8]:
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

In [9]:
tuner.search(
    train_images,
    train_labels,
    validation_split=0.15,
    epochs=15,
    batch_size=128,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

Trial 10 Complete [00h 01m 35s]
val_accuracy: 0.9915555715560913

Best val_accuracy So Far: 0.9945555329322815
Total elapsed time: 00h 15m 40s


In [10]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print("Best Parameters")

for param in best_hp.values:
    print(param, ":", best_hp.values[param])

Best Parameters
filters_1 : 32
dropout_1 : 0.2
filters_2 : 128
dropout_2 : 0.2
filters_3 : 128
dense_units : 128
l2 : 0.0001
dense_dropout : 0.2
learning_rate : 0.0005


In [11]:
best_model = tuner.get_best_models(1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [12]:
history = best_model.fit(
    train_images,
    train_labels,
    validation_split=0.15,
    epochs=30,
    batch_size=128,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

Epoch 1/30


E0000 00:00:1781542856.985752      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


399/399 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9913 - loss: 0.0462 - val_accuracy: 0.9932 - val_loss: 0.0472 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9906 - loss: 0.0493 - val_accuracy: 0.9898 - val_loss: 0.0732 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9919 - loss: 0.0458 - val_accuracy: 0.9907 - val_loss: 0.0695 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9925 - loss: 0.0456 - val_accuracy: 0.9922 - val_loss: 0.0554 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9952 - loss: 0.0366 - val_accuracy: 0.9947 - val_loss: 0.0447 - learning_rate: 2.5000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9955 - loss: 0.0336 - val_accuracy: 0.9944 - val_loss: 0.0441 - learning_rate: 2.5000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.99

In [13]:
test_loss, test_acc = best_model.evaluate(
    test_images,
    test_labels
)

print("Test Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9960 - loss: 0.0279
Test Accuracy: 0.9959999918937683
